# 06 — Feature Selection Analysis
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Identify which features carry the most discriminative information and which are redundant.

> Too many correlated features confuse models — they double-count the same signal.  
> Too few features miss important patterns. Feature selection finds the right balance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)

PALETTE = ['#1f77b4','#2ca02c','#d62728','#9467bd','#8c564b',
           '#e377c2','#7f7f7f','#bcbd22','#17becf','#ff7f0e']

# Load the engineered dataset produced in Section 05
tel_fe = pd.read_csv('processed_v2/telemetry_engineered.csv', parse_dates=['timestamp'])
print('Loaded engineered dataset:', tel_fe.shape)

In [ ]:
# Define the numeric columns to analyse
# (timestamp, parameter are non-numeric -- excluded)
NUMERIC_COLS = ['value','hour','minute','weekday','elapsed_sec',
                'minute_of_day','rolling_mean_5','rolling_std_5',
                'rolling_mean_10','deviation_from_mean',
                'change_rate','abs_change_rate',
                'lag_1','lag_2','lag_3','z_score']

# Use OBC_TEMP as the example parameter for within-parameter correlation analysis
# (Cross-parameter analysis comes later in 6.5)
sample = tel_fe[tel_fe['parameter'] == 'OBC_TEMP'][NUMERIC_COLS].dropna()
print('Sample parameter: OBC_TEMP — rows:', len(sample), '| columns:', len(NUMERIC_COLS))

### 6.1 Correlation Matrix — OBC_TEMP Features
Pearson correlation between all engineered features. Values close to ±1 = redundant pair.

In [ ]:
corr = sample.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Show only lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.4, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — OBC_TEMP Features  (dark red/blue = strongly correlated)',
             fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('plots_v2/06a_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS (to explain to guide):
# - value, rolling_mean_5, rolling_mean_10, lag_1, lag_2, lag_3 are HIGHLY correlated
#   because lag and rolling mean are just smoothed/delayed versions of the raw value
#   --> For classical ML: keep only value + rolling_mean_5 (drop the rest as redundant)
#   --> For deep learning autoencoders: keep all (they provide rich reconstruction context)
# - rolling_std_5, change_rate, abs_change_rate, deviation_from_mean are LESS correlated
#   with value -- they carry INDEPENDENT information about volatility and rate
#   --> These are the MOST VALUABLE features for anomaly detection
# - z_score correlates with deviation_from_mean (it's just the normalised version)
# - temporal features (hour, weekday) show LOW correlation with value -- independent context

### 6.2 High-Correlation Pairs (|r| > 0.90)
Feature pairs above this threshold are candidates for removal in classical ML pipelines.

In [ ]:
THRESHOLD = 0.90
pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        v = corr.iloc[i, j]
        if abs(v) > THRESHOLD:
            pairs.append({'Feature A': corr.columns[i],
                          'Feature B': corr.columns[j],
                          'Correlation': round(v, 4)})

if pairs:
    hc = pd.DataFrame(pairs).sort_values('Correlation', key=abs, ascending=False)
    display(hc)
    print(f'\n--> {len(hc)} redundant pairs identified (|r| > {THRESHOLD})')
    print('    For classical ML, pick ONE from each pair to avoid multicollinearity.')
    print('    For GRU/TCN autoencoders, all features can be kept.')
else:
    print(f'No pairs exceed |r| = {THRESHOLD}')

### 6.3 Variance Analysis
Features with near-zero variance carry no signal — they're essentially constant.

In [ ]:
var_df = sample.var().sort_values(ascending=False).reset_index()
var_df.columns = ['Feature', 'Variance']

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(var_df['Feature'], var_df['Variance'],
        color=[PALETTE[i % len(PALETTE)] for i in range(len(var_df))])
ax.set_xscale('log')  # Log scale because variances span many orders of magnitude
ax.set_title('Feature Variance (log scale) — OBC_TEMP  (longer bar = more informative)',
             fontweight='bold')
ax.set_xlabel('Variance (log scale)')
ax.axvline(0.001, color='red', ls='--', lw=1.2, label='Low-variance threshold (drop candidates)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots_v2/06b_variance.png', dpi=150, bbox_inches='tight')
plt.show()

display(var_df.round(6))

# KEY RESULTS:
# - elapsed_sec has the highest variance (it grows monotonically over time -- expected)
# - minute_of_day also high (spans 0-1439)
# - rolling_std_5, change_rate, abs_change_rate have moderate variance -- MOST USEFUL
#   for anomaly detection (they specifically measure deviation from normal behaviour)
# - Features to the left of red line (if any) should be dropped
#   -- a constant feature adds no discriminative power to any model

### 6.4 Recommended Feature Sets per Model

In [ ]:
rec = pd.DataFrame({
    'Model Type': [
        'Classical ML  (Isolation Forest, One-Class SVM)',
        'Deep Learning (GRU / TCN Autoencoder)',
        'NCDE (irregular time-series)',
    ],
    'Recommended Features': [
        'value, rolling_mean_5, rolling_std_5, deviation_from_mean, '
        'change_rate, abs_change_rate, z_score, lag_1, hour, minute_of_day',
        'All features including lag_1/2/3, rolling_mean_5/10 '
        '(richer reconstruction context helps autoencoder learn normal patterns better)',
        'value, timestamp, elapsed_sec, temporal features only '
        '(model handles continuity and irregular spacing natively -- no need for manual lags)',
    ],
    'Why Drop Others': [
        'lag_2/3 and rolling_mean_10 redundant with lag_1 and rolling_mean_5',
        'Only drop near-zero variance features',
        'Rolling/lag features are computed internally by the ODE solver',
    ]
})
display(rec)

### 6.5 Cross-Parameter Correlation — All 50 Parameters
In wide format, we can see how parameters correlate *with each other* — revealing physical relationships.

In [ ]:
# Pivot to wide format to compute cross-parameter correlation
wide = tel_fe.pivot_table(index='timestamp', columns='parameter',
                           values='value', aggfunc='mean')
wide.columns.name = None
wide = wide.ffill().bfill()  # fill NaN before correlating

cross_corr = wide.corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cross_corr, annot=False, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.2,
            cbar_kws={'shrink': 0.6})
ax.set_title('Cross-Parameter Correlation — All 50 Parameters\n'
             '(dark clusters = subsystem parameters that move together)',
             fontweight='bold')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig('plots_v2/06c_cross_param_corr.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS:
# - Look for dark red/blue clusters -- these are subsystem groups that co-vary
# - BATT_VOLTAGE_1 and BATT_VOLTAGE_2 should show strong positive correlation
#   (both batteries charge/discharge together)
# - GYRO_X/Y/Z may show cross-axis coupling if spacecraft manoeuvres affect all axes
# - SOLAR_POWER_TOTAL and BUS_VOLTAGE should be positively correlated
#   (more solar = higher bus voltage)
# - This cross-correlation structure is what GRU/TCN autoencoders will learn to reconstruct
#   --> An anomaly that breaks these natural correlations will produce high reconstruction error